# RADR mk.60 — Performance Dashboard

**Notional · NOT live-fire**

| Machine | Set `N_MC` in Monte Carlo cell |
|---------|-------------------------------|
| Laptop | **200** (default) |
| RunPod 4090 pod | **5000+** (CPU math; GPU not required) |

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def find_repo_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path("..").resolve()]:
        if (p / "scripts" / "radr_performance_model.py").exists():
            return p.resolve()
    raise FileNotFoundError("Clone RADR-mk.60 and run from repo or notebooks/")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"
DATA = ROOT / "data"
OUT = (ROOT / "notebooks" / "output") if (ROOT / "notebooks").exists() else Path("output")
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPTS))

print("ROOT:", ROOT)
print("OUT:", OUT)

In [ ]:
# Regenerate model JSON (fast on CPU)
subprocess.run(
    [sys.executable, str(SCRIPTS / "radr_performance_model.py"), "--verify"],
    cwd=ROOT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        str(SCRIPTS / "radr_performance_model.py"),
        "--json-out",
        str(DATA / "performance_model_output.json"),
    ],
    cwd=ROOT,
    check=True,
)
report = json.loads((DATA / "performance_model_output.json").read_text(encoding="utf-8"))
print("Model v", report["model_version"], "| impulse", report["motor"]["impulse_ns"], "N*s")

In [ ]:
# Title card — range envelope
ranges = report["ranges"]
keys = sorted(ranges.keys(), key=int)
tof = [ranges[k]["tof_s"] for k in keys]
vel = [ranges[k]["v_mps"] for k in keys]
xm = [int(k) for k in keys]

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(xm, vel, "o-", color="#2d5a27", lw=2, markersize=8, label="v @ range")
ax1.axhspan(330, 350, color="#c4e7c4", alpha=0.5, label="Locked band @ 1000 m")
ax1.axvline(1000, color="#8b0000", ls="--", alpha=0.7, label="Sweet spot")
ax1.axvline(200, color="#555", ls=":", alpha=0.7, label="Min 200 m")
ax1.axvline(1200, color="#555", ls="-.", alpha=0.7, label="Max 1200 m")
ax1.set_xlabel("Downrange (m)")
ax1.set_ylabel("Velocity (m/s)")
ax1.set_title("RADR mk.60 — Velocity along trajectory (nominal)")
ax1.legend(loc="lower right", fontsize=8)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(xm, tof, "s--", color="#1a4d7a", alpha=0.8, label="TOF")
ax2.set_ylabel("Time of flight (s)")
fig.tight_layout()
fig.savefig(OUT / "range_envelope.png", dpi=150)
plt.show()
print("Saved", OUT / "range_envelope.png")

In [ ]:
# Sensitivity @ 1000 m
sens = report["sensitivity_1000m"]
names = [r["case"] for r in sens]
deltas = [r["delta_v_mps"] for r in sens]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2d5a27" if d >= 0 else "#8b4513" for d in deltas]
ax.barh(names, deltas, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Delta v @ 1000 m (m/s)")
ax.set_title("Sensitivity — velocity at sweet spot")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "sensitivity_1000m.png", dpi=150)
plt.show()

In [ ]:
# Evasion geometry (why +200 m does not matter much)
ev = report["evasion_geometry"]
speeds = [r["drone_speed_mps"] for r in ev]
extra = [r["extra_lateral_m_1000_to_1200m"] for r in ev]
full = [r["lateral_m_during_1000m_tof"] for r in ev]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(speeds, extra, "o-", label="Extra lateral 1000→1200 m (~0.6 s)")
ax.plot(speeds, full, "s--", alpha=0.6, label="Lateral during full 1000 m TOF")
ax.set_xlabel("Target speed (m/s)")
ax.set_ylabel("Lateral displacement (m)")
ax.set_title("Maneuver margin — extra 200 m downrange")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "evasion_geometry.png", dpi=150)
plt.show()

In [ ]:
# Monte Carlo envelope @ 1000 m — raise N_MC on RunPod
N_MC = 200  # RunPod: 5000–10000

from monte_carlo_envelope import run_mc

mc = run_mc(N_MC, seed=42, target_m=1000.0)
print(json.dumps(mc, indent=2))

# Re-run with stored samples for histogram (import internals)
from radr_performance_model import CDA_COAST_M2, ModelConfig, simulate

rng = np.random.default_rng(42)
v_samples = []
for _ in range(N_MC):
    mass = rng.uniform(2.95, 3.35)
    impulse_scale = rng.uniform(2950 / 3000, 3050 / 3000)
    loft = rng.uniform(2.5, 5.0)
    cda_coast = CDA_COAST_M2 * rng.uniform(0.9, 1.1)
    cfg = ModelConfig(
        mass_kg=mass,
        loft_deg=loft,
        thrust_scale=impulse_scale,
        cda_coast_m2=cda_coast,
    )
    v_samples.append(simulate(cfg, [1000.0]).ranges["1000"].v_mps)

v_samples = np.array(v_samples)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(v_samples, bins=min(40, N_MC // 5), color="#2d5a27", edgecolor="white", alpha=0.85)
ax.axvline(330, color="red", ls="--", label="330 m/s")
ax.axvline(350, color="red", ls="--", label="350 m/s")
ax.axvline(mc["v_mps"]["p50"], color="navy", lw=2, label=f"p50={mc['v_mps']['p50']:.1f}")
ax.set_xlabel("v @ 1000 m (m/s)")
ax.set_title(f"Monte Carlo (n={N_MC}) — mass, impulse, loft, coast drag")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "monte_carlo_v1000.png", dpi=150)
plt.show()
print(f"In-band fraction: {mc['fraction_in_330_350_band']*100:.1f}%")

## Pitch exports

Figures saved under `notebooks/output/`:

- `range_envelope.png`
- `sensitivity_1000m.png`
- `evasion_geometry.png`
- `monte_carlo_v1000.png`

**Locked story:** 200 m min · 800–1200 m band · **1000 m sweet spot** · 300 cubes · tank-shell load · ~335 m/s @ 1000 m (nominal model).